# 10 — Measuring RAG Instead of Eyeballing It

Our RAG system works. How do we actually *know* that, beyond "I tried a few questions and they looked right"?

That's fine for 10 questions. It falls apart at 1,000 — and it completely falls apart the moment you change something (a chunk size, an embedding model, a prompt) and need to know if things got better or worse. Eyeballing doesn't scale. Numbers do.


## Three ways RAG can go wrong

```
Question -> [ Retrieval ] -> Chunks -> [ Generation ] -> Answer

Failure 1: wrong chunks retrieved           (retrieval failure)
Failure 2: right chunks, wrong answer       (generation failure)
Failure 3: right-sounding answer, not       (hallucination, even with RAG)
           actually grounded in the chunks
```

Each failure needs its own metric — "the answer looked fine" doesn't tell you which of these three broke.


In [ ]:
import os
import json
import faiss
import numpy as np
from dotenv import load_dotenv
from groq import Groq
from sentence_transformers import SentenceTransformer

load_dotenv()

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-20b"
embedder = SentenceTransformer("all-MiniLM-L6-v2")

DATA_DIR = os.path.join("..", "data")
index = faiss.read_index(os.path.join(DATA_DIR, "storyverse_chunks.faiss"))
with open(os.path.join(DATA_DIR, "storyverse_chunks_docstore.json")) as f:
    doc_store = json.load(f)


def retrieve(query: str, top_k: int = 3) -> list[dict]:
    query_vec = embedder.encode([query]).astype(np.float32)
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, top_k)
    return [{**doc_store[i], "score": float(scores[0][j])} for j, i in enumerate(indices[0])]


def rag_answer(question: str, top_k: int = 3) -> dict:
    chunks = [r for r in retrieve(question, top_k) if r["score"] >= 0.35]
    if not chunks:
        return {"answer": "I don't have enough information about this in my knowledge base.", "chunks": []}
    context = "\n\n".join(f"[{c['metadata']['source']}]\n{c['content']}" for c in chunks)
    prompt = f"Answer using ONLY this context.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    response = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0.1)
    return {"answer": response.choices[0].message.content, "chunks": chunks}


## Building a ground-truth eval set

Ten question-answer pairs, two per StoryVerse story, each with the source file we *expect* retrieval to surface. This is your regression test suite. Build it once, before you change anything.


In [ ]:
eval_dataset = [
    {"question": "Who is Cooper's daughter in Interstellar?", "expected_answer": "Murph (Murphy)", "expected_source": "interstellar.txt"},
    {"question": "How does Cooper save Earth?", "expected_answer": "By sending gravitational data through the tesseract", "expected_source": "interstellar.txt"},
    {"question": "What object is Voldemort trying to steal?", "expected_answer": "The Sorcerer's Stone", "expected_source": "harry_potter_sorcerers_stone.txt"},
    {"question": "Who are Harry's two closest friends?", "expected_answer": "Ron Weasley and Hermione Granger", "expected_source": "harry_potter_sorcerers_stone.txt"},
    {"question": "Who does Shivudu fall in love with?", "expected_answer": "Avanthika", "expected_source": "baahubali.txt"},
    {"question": "Who betrayed Shivudu's father?", "expected_answer": "Bhallaladeva", "expected_source": "baahubali.txt"},
    {"question": "What happens to Arjun in the bookshop?", "expected_answer": "He steps through a portal to a beach under two moons", "expected_source": "portal_bookshop.txt"},
    {"question": "Where is the bookshop from Arjun's story located?", "expected_answer": "Hyderabad's Abids market", "expected_source": "portal_bookshop.txt"},
    {"question": "Why does Batman take the blame for Harvey Dent's crimes?", "expected_answer": "To protect Gotham's faith in its 'white knight'", "expected_source": "dark_knight.txt"},
    {"question": "Who is the criminal mastermind causing chaos in Gotham?", "expected_answer": "The Joker", "expected_source": "dark_knight.txt"},
]

print(f"Eval set: {len(eval_dataset)} questions")


## Metric 1 — retrieval precision

Did we retrieve the right *document* at all? A blunt but essential check: is the expected source anywhere in the top-k results?


In [ ]:
def retrieval_precision(question: str, expected_source: str, top_k: int = 3) -> int:
    results = retrieve(question, top_k)
    retrieved_sources = [r["metadata"]["source"] for r in results]
    return 1 if expected_source in retrieved_sources else 0


scores = [retrieval_precision(item["question"], item["expected_source"]) for item in eval_dataset]
print(f"Retrieval precision: {sum(scores)}/{len(scores)}")


If this number is low, the fix is in retrieval — chunk size, embedding model, metadata — not the model doing the answering. Don't touch the prompt until this number looks right.


## Metric 2 — answer faithfulness

Is the answer actually grounded in the retrieved chunks, or did the model wander off and make something up anyway? We use the model itself as a judge.


In [ ]:
def check_faithfulness(answer: str, context: str) -> str:
    prompt = f"""Given this context:
{context}

And this answer:
{answer}

Does the answer contain ONLY information present in the context?
Reply with exactly one word first -- FAITHFUL or UNFAITHFUL -- then a one-line reason."""
    response = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0.0)
    return response.choices[0].message.content


sample = eval_dataset[0]
result = rag_answer(sample["question"])
context = "\n\n".join(c["content"] for c in result["chunks"])
print(check_faithfulness(result["answer"], context))


**LLM-as-judge** is imperfect -- it can be wrong, and it's another model call to trust -- but it's scalable in a way manual review never is. For anything safety-critical, pair it with periodic human spot checks.


## Metric 3 — answer correctness

Is the answer actually right, compared to our ground truth?


In [ ]:
def check_correctness(answer: str, expected_answer: str) -> str:
    prompt = f"""Expected answer: {expected_answer}
Actual answer: {answer}

Is the actual answer correct and equivalent to the expected answer?
Reply with exactly one label first -- CORRECT, PARTIALLY_CORRECT, or INCORRECT -- then a one-line reason."""
    response = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0.0)
    return response.choices[0].message.content


print(check_correctness(result["answer"], sample["expected_answer"]))


## The regression test runner

All three metrics, run across the entire eval set, in one call. This is what you re-run after every change to chunk size, prompt wording, or embedding model.


In [ ]:
def run_evaluation(eval_dataset: list[dict]) -> list[dict]:
    results = []
    for item in eval_dataset:
        precision = retrieval_precision(item["question"], item["expected_source"])
        rag_result = rag_answer(item["question"])
        context = "\n\n".join(c["content"] for c in rag_result["chunks"])
        faithfulness = check_faithfulness(rag_result["answer"], context)
        correctness = check_correctness(rag_result["answer"], item["expected_answer"])
        results.append({
            "question": item["question"],
            "precision": precision,
            "faithfulness": faithfulness.split()[0] if faithfulness else "N/A",
            "correctness": correctness.split()[0] if correctness else "N/A",
        })
    return results


evaluation_results = run_evaluation(eval_dataset)
for r in evaluation_results:
    print(f"[precision={r['precision']}] [{r['faithfulness']:<11}] [{r['correctness']:<17}] {r['question']}")

precision_total = sum(r["precision"] for r in evaluation_results)
print(f"\nOverall retrieval precision: {precision_total}/{len(evaluation_results)}")


Run this exact cell before and after every change you make. If the numbers drop, revert the change -- you now have a number telling you so, instead of a hunch.


## Setting up LangSmith

**LangSmith** is an observability platform for LLM apps -- think of it as the equivalent of an APM tool (like Datadog), but for tracing prompts, retrieved chunks, and model calls instead of HTTP requests.


In [ ]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.environ.get("LANGSMITH_API_KEY", "")
os.environ["LANGCHAIN_PROJECT"] = "storyverse-ai"

from langchain_classic.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

lc_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
llm = ChatGroq(model=MODEL, temperature=0.1)

chroma_store = Chroma.from_texts(
    texts=[d["content"] for d in doc_store],
    embedding=lc_embeddings,
    metadatas=[d["metadata"] for d in doc_store],
    collection_name="storyverse_eval",
)
qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=chroma_store.as_retriever(search_kwargs={"k": 3}))

result = qa_chain.invoke({"query": "Who is Cooper's daughter in Interstellar?"})
print(result["result"])
print("\nCheck your LangSmith project 'storyverse-ai' -- this call should now appear there as a trace.")


Open that trace in the LangSmith dashboard and you'll see, step by step: the input question, the retriever call and exactly which chunks it returned, the full prompt sent to the model, the model's raw output, and the latency of each individual step.


## Debugging a bad answer with a trace

Say a user reports: "It told me Cooper's daughter was Jessica, but it's Murph." Here's how you'd actually debug that in production, instead of guessing:

1. Open the LangSmith trace for that exact question.
2. Check what the retriever actually returned. **Wrong chunk retrieved?** That's a retrieval problem — go fix chunking or the embedding model, same as Metric 1 above catches.
3. **Right chunk retrieved, but the model still said Jessica?** That's a generation problem — the prompt isn't forcing grounding hard enough, or the chunk's actual wording is more ambiguous than you assumed.

This is how you debug RAG in production: by tracing the actual retrieval and generation steps, not by staring at the final answer and guessing which half broke.


## What to actually track in production

| Metric | What it catches | How to measure |
|---|---|---|
| Retrieval precision | Wrong chunks retrieved | Expected source in top-k? |
| Answer faithfulness | Hallucination despite RAG | LLM-as-judge against retrieved context |
| Answer correctness | Wrong answers overall | LLM-as-judge against ground truth |
| Latency | Slow responses | `time.time()`, or read it straight off a LangSmith trace |
| Token cost | Expensive context | `response.usage.total_tokens` |

Start with precision and correctness — they're the cheapest to compute and catch the most common failures. Add faithfulness once hallucination specifically becomes a concern worth tracking on its own.


## The series, start to finish

- **00** — why RAG exists: hallucination, training cutoffs, context limits
- **01-02** — manual context injection, then moving knowledge into files
- **03** — why sending everything fails, measured
- **04-05** — embeddings and vector search, built from scratch
- **06** — chunking strategies
- **07** — the full retrieval pipeline
- **08** — complete, grounded RAG with source attribution
- **09** — the failure modes production systems actually hit
- **10** — measuring quality instead of guessing at it

You now understand RAG from the inside out -- not just how to call `RetrievalQA.from_chain_type()`, but what it's doing underneath and why every piece exists.

**What's next**, as a separate, more advanced series: conversational RAG with memory, agentic RAG with tool use and routing, and multi-modal RAG over images and PDFs.

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| LLM-as-judge | Using a model to grade another model's answer |
| Retrieval precision | Whether the right source document showed up in the results |
| Faithfulness | Whether an answer sticks to what the retrieved context actually says |
| Tracing (LangSmith) | Recording every step of a pipeline run for later inspection |

**Exercises:**

- Add 5 more questions to `eval_dataset` and re-run `run_evaluation`.
- Deliberately shrink your chunk size to something too small (from notebook 06) and re-run the eval -- watch retrieval precision drop.
- Swap `all-MiniLM-L6-v2` for a larger embedding model and check whether correctness actually improves, or just latency gets worse.
